In [2]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

In [3]:
df = pd.read_csv('../data/processed/ratings_clean.csv')
movies = pd.read_csv('../data/raw/movies.csv')
tags = pd.read_csv('../data/raw/tags.csv')

print(df.shape)

(100836, 6)


In [4]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
split_date = df['timestamp'].quantile(0.8)

In [5]:
train_df = df[df['timestamp'] <= split_date]
test_df  = df[df['timestamp'] > split_date]

print(f"Train: {len(train_df):,} ratings")
print(f"Test:  {len(test_df):,} ratings")
print(f"Split date: {split_date}")

Train: 80,669 ratings
Test:  20,167 ratings
Split date: 2016-03-22 08:26:11


In [6]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)

trainset = data.build_full_trainset()

svd = SVD(n_factors=100, random_state=42)

svd.fit(trainset)
print("Model trained!")

Model trained!


In [7]:
def precision_at_k(user_id, k=10, threshold=3.5):
    # get test movies for this user
    user_test = test_df[test_df['userId'] == user_id]
    if len(user_test) == 0:
        return None
    
    # relevant movies = rated above threshold in test set
    relevant = set(user_test[user_test['rating'] >= threshold]['movieId'].tolist())
    if len(relevant) == 0:
        return None

    # get top K predictions
    rated_movies = train_df[train_df['userId'] == user_id]['movieId'].tolist()
    all_movies = df['movieId'].unique()
    unrated = [m for m in all_movies if m not in rated_movies]

    predictions = [svd.predict(user_id, m) for m in unrated]
    predictions.sort(key=lambda x: x.est, reverse=True)
    top_k = [p.iid for p in predictions[:k]]

    hits = len(set(top_k) & relevant)
    return hits / k



In [8]:
def ndcg_at_k(user_id, k=10, threshold=3.5):
    user_test = test_df[test_df['userId'] == user_id]
    if len(user_test) == 0:
        return None

    relevant = set(user_test[user_test['rating'] >= threshold]['movieId'].tolist())
    if len(relevant) == 0:
        return None

    rated_movies = train_df[train_df['userId'] == user_id]['movieId'].tolist()
    all_movies = df['movieId'].unique()
    unrated = [m for m in all_movies if m not in rated_movies]

    predictions = [svd.predict(user_id, m) for m in unrated]
    predictions.sort(key=lambda x: x.est, reverse=True)
    top_k = [p.iid for p in predictions[:k]]

    dcg = sum([1 / np.log2(i + 2) for i, m in enumerate(top_k) if m in relevant])
    idcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / idcg if idcg > 0 else 0

In [9]:
sample_users = test_df['userId'].unique()[:50]

precisions, ndcgs = [], []

for user_id in sample_users:
    p = precision_at_k(user_id, k=10)
    n = ndcg_at_k(user_id, k=10)
    if p is not None:
        precisions.append(p)
    if n is not None:
        ndcgs.append(n)

print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"NDCG@10:      {np.mean(ndcgs):.4f}")

Precision@10: 0.1680
NDCG@10:      0.2005


In [10]:
results = {
    'model': ['SVD (Collaborative Filtering)'],
    'precision@10': [round(np.mean(precisions), 4)],
    'ndcg@10': [round(np.mean(ndcgs), 4)]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                        model  precision@10  ndcg@10
SVD (Collaborative Filtering)         0.168   0.2005


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tags_clean = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
tags_clean.columns = ['movieId', 'tags']
movies = movies.merge(tags_clean, on='movieId', how='left')
movies['tags'] = movies['tags'].fillna('')
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['features'] = movies['genres_clean'] + ' ' + movies['tags']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['features'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Content-based model ready!")

Content-based model ready!


In [12]:
def precision_at_k_cb(user_id, k=10, threshold=3.5):
    user_test = test_df[test_df['userId'] == user_id]
    if len(user_test) == 0:
        return None

    relevant = set(user_test[user_test['rating'] >= threshold]['movieId'].tolist())
    if len(relevant) == 0:
        return None

    # get user's top 5 rated movies from training set
    top5 = (train_df[train_df['userId'] == user_id]
            .sort_values('rating', ascending=False)
            .head(5)['movieId'].tolist())

    if len(top5) == 0:
        return None

    # average similarity scores across top 5
    rated_movies = train_df[train_df['userId'] == user_id]['movieId'].tolist()
    scores = {}
    for movie_id in movies['movieId']:
        if movie_id in rated_movies:
            continue
        idx = movies[movies['movieId'] == movie_id].index
        if len(idx) == 0:
            continue
        idx = idx[0]
        cb_scores = []
        for rated_id in top5:
            rated_idx = movies[movies['movieId'] == rated_id].index
            if len(rated_idx) > 0:
                cb_scores.append(cosine_sim[idx][rated_idx[0]])
        scores[movie_id] = np.mean(cb_scores) if cb_scores else 0

    top_k = sorted(scores, key=scores.get, reverse=True)[:k]
    hits = len(set(top_k) & relevant)
    return hits / k

precisions_cb = []
for user_id in sample_users:
    p = precision_at_k_cb(user_id, k=10)
    if p is not None:
        precisions_cb.append(p)

print(f"Precision@10 (Content-Based): {np.mean(precisions_cb):.4f}")

Precision@10 (Content-Based): 0.0273


In [13]:
results['model'].append('Content-Based')
results['precision@10'].append(round(np.mean(precisions_cb), 4))
results['ndcg@10'].append('-')

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                        model  precision@10 ndcg@10
SVD (Collaborative Filtering)        0.1680  0.2005
                Content-Based        0.0273       -


In [14]:
def precision_at_k_hybrid(user_id, k=10, threshold=3.5, alpha=0.7):
    user_test = test_df[test_df['userId'] == user_id]
    if len(user_test) == 0:
        return None

    relevant = set(user_test[user_test['rating'] >= threshold]['movieId'].tolist())
    if len(relevant) == 0:
        return None

    rated_movies = train_df[train_df['userId'] == user_id]['movieId'].tolist()
    all_movies = df['movieId'].unique()
    unrated = [m for m in all_movies if m not in rated_movies]

    top5 = (train_df[train_df['userId'] == user_id]
            .sort_values('rating', ascending=False)
            .head(5)['movieId'].tolist())

    scores = {}
    for movie_id in unrated:
        cf_score = svd.predict(user_id, movie_id).est

        idx = movies[movies['movieId'] == movie_id].index
        if len(idx) == 0:
            continue
        idx = idx[0]

        cb_scores = []
        for rated_id in top5:
            rated_idx = movies[movies['movieId'] == rated_id].index
            if len(rated_idx) > 0:
                cb_scores.append(cosine_sim[idx][rated_idx[0]])

        cb_score = np.mean(cb_scores) if cb_scores else 0
        scores[movie_id] = alpha * cf_score + (1 - alpha) * cb_score

    top_k = sorted(scores, key=scores.get, reverse=True)[:k]
    hits = len(set(top_k) & relevant)
    return hits / k

precisions_hybrid = []
for user_id in sample_users:
    p = precision_at_k_hybrid(user_id, k=10)
    if p is not None:
        precisions_hybrid.append(p)

print(f"Precision@10 (Hybrid): {np.mean(precisions_hybrid):.4f}")

Precision@10 (Hybrid): 0.1680


In [15]:
results['model'].append('Hybrid')
results['precision@10'].append(round(np.mean(precisions_hybrid), 4))
results['ndcg@10'].append('-')

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                        model  precision@10 ndcg@10
SVD (Collaborative Filtering)        0.1680  0.2005
                Content-Based        0.0273       -
                       Hybrid        0.1680       -


In [16]:
results_df.to_csv('../data/processed/model_comparison.csv', index=False)
print("Saved!")

Saved!


In [17]:
from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

print(f"Best RMSE: {gs.best_score['rmse']:.4f}")
print(f"Best params: {gs.best_params['rmse']}")

Best RMSE: 0.8693
Best params: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [18]:
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5.0))

train_data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
trainset = train_data.build_full_trainset()

# build testset from test_df
testset = list(zip(test_df['userId'], test_df['movieId'], test_df['rating']))

best_svd = SVD(n_factors=150, n_epochs=30, lr_all=0.01, reg_all=0.1, random_state=42)
best_svd.fit(trainset)

predictions = best_svd.test(testset)
print(f"Tuned RMSE: {accuracy.rmse(predictions):.4f}")
print(f"Tuned MAE:  {accuracy.mae(predictions):.4f}")

RMSE: 1.0055
Tuned RMSE: 1.0055
MAE:  0.7766
Tuned MAE:  0.7766


In [19]:
precisions_tuned = []
for user_id in sample_users:
    p = precision_at_k(user_id, k=10)
    if p is not None:
        precisions_tuned.append(p)

print(f"Precision@10 (Tuned SVD): {np.mean(precisions_tuned):.4f}")

Precision@10 (Tuned SVD): 0.1680


In [20]:
results = {
    'model': ['SVD (default)', 'SVD (tuned)', 'Content-Based', 'Hybrid'],
    'rmse': [0.8807, 1.0055, '-', '-'],
    'precision@10': [0.1680, 0.1680, 0.0273, 0.1680],
    'ndcg@10': [0.2005, '-', '-', '-'],
    'split': ['random', 'temporal', 'temporal', 'temporal']
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv('../data/processed/model_comparison.csv', index=False)

        model    rmse  precision@10 ndcg@10    split
SVD (default)  0.8807        0.1680  0.2005   random
  SVD (tuned)  1.0055        0.1680       - temporal
Content-Based       -        0.0273       - temporal
       Hybrid       -        0.1680       - temporal


In [21]:
ratings_25m = pd.read_csv('../data/raw/ml-25m/ratings.csv')
movies_25m = pd.read_csv('../data/raw/ml-25m/movies.csv')
tags_25m = pd.read_csv('../data/raw/ml-25m/tags.csv')

print(ratings_25m.shape, movies_25m.shape, tags_25m.shape)

(25000095, 4) (62423, 3) (1093360, 4)


In [22]:
import time

reader = Reader(rating_scale=(0.5, 5.0))
data_25m = Dataset.load_from_df(ratings_25m[['userId', 'movieId', 'rating']], reader)
trainset_25m = data_25m.build_full_trainset()

start = time.time()
svd_25m = SVD(n_factors=150, n_epochs=30, lr_all=0.01, reg_all=0.1, random_state=42)
svd_25m.fit(trainset_25m)
print(f"Training time: {time.time() - start:.0f} seconds")

Training time: 410 seconds


In [23]:
ratings_25m['timestamp'] = pd.to_datetime(ratings_25m['timestamp'], unit='s')
split_date = ratings_25m['timestamp'].quantile(0.8)

train_25m = ratings_25m[ratings_25m['timestamp'] <= split_date]
test_25m = ratings_25m[ratings_25m['timestamp'] > split_date]

testset_25m = list(zip(test_25m['userId'], test_25m['movieId'], test_25m['rating']))

predictions_25m = svd_25m.test(testset_25m)
print(f"RMSE: {accuracy.rmse(predictions_25m):.4f}")
print(f"MAE:  {accuracy.mae(predictions_25m):.4f}")

RMSE: 0.8010
RMSE: 0.8010
MAE:  0.6017
MAE:  0.6017


In [24]:
import pickle

with open('../data/processed/svd_25m.pkl', 'wb') as f:
    pickle.dump(svd_25m, f)

print("Model saved!")

Model saved!


In [25]:
df_25m = ratings_25m.merge(movies_25m, on='movieId')
df_25m.to_csv('../data/processed/ratings_25m_clean.csv', index=False)
print(f"Saved {len(df_25m):,} rows")

Saved 25,000,095 rows


In [26]:
import pickle
from surprise import SVD, Dataset, Reader
import pandas as pd

ratings_25m = pd.read_csv('../data/processed/ratings_25m_clean.csv')

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_25m[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=150, n_epochs=30, lr_all=0.01, reg_all=0.1, random_state=42)
svd.fit(trainset)

with open('../data/processed/svd_25m.pkl', 'wb') as f:
    pickle.dump(svd, f)

print("Saved!")

Saved!
